# 🧹 Carlist.my Data Cleaning Pipeline
---
**Source:** BeautifulSoup scrape of carlist.my (100,005 rows × 11 columns)  
**Goal:** Produce a clean, analysis-ready `carlist_cleaned.csv`

### Column inventory
| Column | Raw Format | Issue |
|---|---|---|
| `price` | `RM 63,999` | String → numeric |
| `monthly_payment` | `RM 830 / month` | String → numeric, 209 nulls |
| `mileage` | `75 - 80K KM` | Range string → midpoint numeric |
| `seller_type` | Noisy long text | Extract clean label |
| `location` | `State , City` | Split into two columns |
| `year` | int | Filter outliers |


In [ ]:
# ── 1. Imports & Setup ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import time
import tracemalloc
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.0f}'.format)

print("✅ Imports OK")
print(f"Pandas {pd.__version__} | NumPy {np.__version__}")


In [ ]:
# ── 2. Load Raw Data ────────────────────────────────────────────────────────
tracemalloc.start()
t0 = time.perf_counter()

RAW_PATH = '/mnt/user-data/uploads/carlist_100k.csv'
df_raw = pd.read_csv(RAW_PATH)

load_time = time.perf_counter() - t0
mem_current, mem_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"✅ Loaded in {load_time:.3f}s")
print(f"   Shape      : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"   Memory used: {mem_peak / 1e6:.1f} MB (peak)")
print()
print(df_raw.dtypes)


In [ ]:
# ── 3. Baseline Quality Report ──────────────────────────────────────────────
print("=" * 55)
print("BASELINE QUALITY REPORT")
print("=" * 55)
print(f"Total rows          : {len(df_raw):,}")
print(f"Duplicate rows      : {df_raw.duplicated().sum():,}")
print()
print("Missing values per column:")
miss = df_raw.isnull().sum()
for col, n in miss.items():
    pct = n / len(df_raw) * 100
    bar = '█' * int(pct) + '░' * (10 - int(pct))
    print(f"  {col:<18} {n:>5} ({pct:5.1f}%) [{bar}]")
print()
print("Year range:", df_raw['year'].min(), "–", df_raw['year'].max())
print("Condition  :", df_raw['condition'].value_counts().to_dict())
print("Transmission:", df_raw['transmission'].value_counts().to_dict())


---
## 🔧 Cleaning Steps
### Step 1 – Remove duplicates

In [ ]:
# ── 4. Remove Duplicates ────────────────────────────────────────────────────
df = df_raw.copy()
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplicates removed : {before - len(df):,}")
print(f"Rows remaining     : {len(df):,}")


### Step 2 – Clean `price` → numeric (MYR)

In [ ]:
# ── 5. Price Cleaning ───────────────────────────────────────────────────────
def parse_price(val):
    """'RM 63,999' → 63999.0"""
    if pd.isna(val):
        return np.nan
    cleaned = re.sub(r'[^\d.]', '', str(val))
    return float(cleaned) if cleaned else np.nan

df['price_myr'] = df['price'].apply(parse_price)

# Sanity-filter: keep prices between RM 1,000 and RM 5,000,000
mask_price = df['price_myr'].between(1_000, 5_000_000)
print(f"Price outliers removed : {(~mask_price).sum():,}")
df = df[mask_price].copy()

print(f"price_myr range : RM {df['price_myr'].min():,.0f} – RM {df['price_myr'].max():,.0f}")
print(f"price_myr median: RM {df['price_myr'].median():,.0f}")


### Step 3 – Clean `monthly_payment` → numeric

In [ ]:
# ── 6. Monthly Payment Cleaning ─────────────────────────────────────────────
def parse_monthly(val):
    """'RM 830 / month' → 830.0"""
    if pd.isna(val):
        return np.nan
    cleaned = re.sub(r'[^\d.]', '', str(val))
    return float(cleaned) if cleaned else np.nan

df['monthly_payment_myr'] = df['monthly_payment'].apply(parse_monthly)

# Fill nulls with median (only 209 nulls ~0.2%)
median_mp = df['monthly_payment_myr'].median()
df['monthly_payment_myr'].fillna(median_mp, inplace=True)

print(f"monthly_payment nulls filled with median: RM {median_mp:,.0f}")
print(f"monthly_payment_myr range: RM {df['monthly_payment_myr'].min():,.0f} – RM {df['monthly_payment_myr'].max():,.0f}")


### Step 4 – Clean `mileage` → midpoint numeric (km)

In [ ]:
# ── 7. Mileage Cleaning ─────────────────────────────────────────────────────
def parse_mileage(val):
    """
    '75 - 80K KM'  → 77500.0   (midpoint)
    '120 - 125K KM'→ 122500.0
    Handles both 'K KM' and plain numbers.
    """
    if pd.isna(val):
        return np.nan
    val = str(val).upper().replace(',', '')
    nums = re.findall(r'[\d.]+', val)
    if not nums:
        return np.nan
    multiplier = 1000 if 'K' in val else 1
    values = [float(n) * multiplier for n in nums[:2]]
    return sum(values) / len(values)

df['mileage_km'] = df['mileage'].apply(parse_mileage)

print(f"mileage_km range : {df['mileage_km'].min():,.0f} – {df['mileage_km'].max():,.0f} km")
print(f"mileage_km median: {df['mileage_km'].median():,.0f} km")
print(f"Null mileage_km  : {df['mileage_km'].isnull().sum()}")


### Step 5 – Clean `seller_type` (extract label only)

In [ ]:
# ── 8. Seller Type Cleaning ─────────────────────────────────────────────────
KNOWN = ['Dealer', 'Sales Agent', 'Private', 'Broker']

def parse_seller(val):
    """Extract first matching label; unknown → 'Other'"""
    if pd.isna(val):
        return 'Unknown'
    for label in KNOWN:
        if label.lower() in str(val).lower():
            return label
    return 'Other'

df['seller_type_clean'] = df['seller_type'].apply(parse_seller)

print("Seller type distribution:")
print(df['seller_type_clean'].value_counts().to_string())


### Step 6 – Split `location` → `state` + `city`

In [ ]:
# ── 9. Location Splitting ────────────────────────────────────────────────────
# Format: 'State , City'  or  'State'
location_split = df['location'].str.split(r'\s*,\s*', n=1, expand=True)
df['state'] = location_split[0].str.strip()
df['city']  = location_split[1].str.strip() if 1 in location_split.columns else ''
df['city'].fillna('', inplace=True)

print("Top 10 states:")
print(df['state'].value_counts().head(10).to_string())


### Step 7 – Filter unrealistic `year` values

In [ ]:
# ── 10. Year Filtering ───────────────────────────────────────────────────────
CURRENT_YEAR = 2026
before = len(df)
df = df[(df['year'] >= 1980) & (df['year'] <= CURRENT_YEAR)].copy()
print(f"Rows removed (year outliers): {before - len(df):,}")
print(f"Year range after filter: {df['year'].min()} – {df['year'].max()}")


### Step 8 – Standardise `condition` & `transmission` (Title Case)

In [ ]:
# ── 11. Standardise Categorical Columns ─────────────────────────────────────
df['condition']    = df['condition'].str.strip().str.title()
df['transmission'] = df['transmission'].str.strip().str.title()

print("condition    :", df['condition'].value_counts().to_dict())
print("transmission :", df['transmission'].value_counts().to_dict())


### Step 9 – Drop redundant raw columns & finalise schema

In [ ]:
# ── 12. Finalise Schema ─────────────────────────────────────────────────────
KEEP_COLS = [
    'id', 'title', 'year',
    'price_myr', 'monthly_payment_myr',
    'mileage_km', 'transmission',
    'seller_type_clean', 'state', 'city',
    'condition', 'url'
]

df_clean = df[KEEP_COLS].rename(columns={'seller_type_clean': 'seller_type'})
df_clean = df_clean.reset_index(drop=True)

print(f"✅ Final shape : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print()
print(df_clean.dtypes)
print()
print(df_clean.head(3).to_string())


In [ ]:
# ── 13. Final Quality Report ─────────────────────────────────────────────────
print("=" * 55)
print("FINAL QUALITY REPORT")
print("=" * 55)
print(f"Total rows remaining : {len(df_clean):,}")
print(f"Rows cleaned away    : {len(df_raw) - len(df_clean):,}")
print()
print("Missing values per column:")
miss_clean = df_clean.isnull().sum()
for col, n in miss_clean.items():
    pct = n / len(df_clean) * 100
    flag = '⚠️' if n > 0 else '✅'
    print(f"  {flag} {col:<22} {n:>5} ({pct:.2f}%)")
print()
print("Numeric summary:")
print(df_clean[['price_myr', 'monthly_payment_myr', 'mileage_km', 'year']].describe().applymap('{:,.1f}'.format))


In [ ]:
# ── 14. Save Cleaned Data ────────────────────────────────────────────────────
OUT_PATH = '/home/claude/carlist_cleaned.csv'
df_clean.to_csv(OUT_PATH, index=False)
print(f"✅ Saved: {OUT_PATH}")
import os
size_mb = os.path.getsize(OUT_PATH) / 1e6
print(f"   File size: {size_mb:.1f} MB")
print(f"   Rows     : {len(df_clean):,}")
print(f"   Columns  : {df_clean.shape[1]}")
